# W-SDA: Colab worker + 自宅Docker

RoPE → SummerDeltaAttention → NoPE のカスタム言語モデルを、HTTPS経由で協調学習します。
自宅側のcoordinatorを先に起動してください。各GPUにモデル全体が必要です。

**実行条件:** 有料Colabで正のcompute unit残高がある環境、許可された専用環境、または自己管理local runtime。
無料managed runtimeの分散ワーカー実行には使用しません。
[公式FAQ](https://research.google.com/colaboratory/faq.html)を確認してください。
Local runtimeは自宅等の計算機を操作する機能で、Colab GPUを追加しません。

このノートブックは未実行の配布用テンプレートです。ソースZIPの学習コードを確認してから実行してください。


In [ ]:
# 利用可能な環境であることを自分で確認してからTrueに変更します。
I_HAVE_AUTHORIZED_DISTRIBUTED_COMPUTE = False
if not I_HAVE_AUTHORIZED_DISTRIBUTED_COMPUTE:
    raise RuntimeError("有料Colabの正のcompute unit残高、または自己管理環境を確認してください。")


## 1. ソースZIPをインストール

配布された `w-sda-lib.zip`、または認証して取得したGitHubのDownload ZIPを選択します。
Privateリポジトリのアクセストークンをコードに埋め込む必要はありません。
自己管理runtimeでは、ZIPのローカルパスを入力することもできます。

In [ ]:
import sys, subprocess, zipfile, stat
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd() / "wsda_notebook"
BASE.mkdir(parents=True, exist_ok=True)

def choose_zip(label):
    try:
        from google.colab import files
    except ImportError:
        return Path(input(label + " のZIPパス: ")).expanduser().resolve()
    print(label)
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".zip")]
    if len(names) != 1:
        raise ValueError("ZIPを1個だけ選択してください")
    return Path(names[0]).resolve()

def safe_extract(archive, destination, max_bytes):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        if sum(m.file_size for m in z.infolist()) > max_bytes:
            raise ValueError("展開後サイズが上限を超えています")
        for member in z.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("安全でないZIPパスです")
        z.extractall(root)

safe_extract(choose_zip("W-SDAのソースZIP"), BASE / "wsda_source", 50 * 1024**2)
candidates = [p.parent for p in (BASE / "wsda_source").rglob("pyproject.toml")
              if (p.parent / "wsda" / "model.py").exists()]
if len(candidates) != 1:
    raise ValueError("W-SDAのプロジェクトルートを一意に特定できません")
SOURCE = candidates[0]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(SOURCE)])


## 2. 準備済みデータ

自宅で `wsda prepare` を実行し、`python -m zipfile -c data.zip data` で作成したZIPを選択します。
同じデータを両ノードで持ち、home=shard 0 / colab=shard 1に分けます。
元のテキストや学習データをColabにアップロードする権限があることを確認してください。

In [ ]:
safe_extract(choose_zip("準備済みdata.zip"), BASE / "wsda_data", 2 * 1024**3)
data_dirs = [p.parent for p in (BASE / "wsda_data").rglob("manifest.json")
             if (p.parent / "train.npy").exists()]
if len(data_dirs) != 1:
    raise ValueError("dataディレクトリを一意に特定できません")
DATA = data_dirs[0]


## 3. 接続先・秘密値・GPU確認

HTTPS URLを入力し、tokenには自宅の `.env` の `WSDA_TOKEN` と同じ値を指定します。
値は表示しません。ノートブックのセルに秘密値を直書きしないでください。
Colab側のGPUランタイムを選択してください。CPUは動作確認用です。

In [ ]:
from getpass import getpass
import torch
from wsda.worker import Client

SERVER = input("coordinatorのHTTPS URL: ").strip()
if not SERVER.startswith("https://"):
    raise ValueError("このノートブックはHTTPS接続を使用します")
TOKEN = getpass("WSDA_TOKEN: ")
print({"torch": torch.__version__, "cuda": torch.cuda.is_available(),
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})
client = Client(SERVER, TOKEN)
try:
    import json
    print(json.loads(client.request("GET", "/v1/status")))
finally:
    client.close()


## 4. 学習ワーカーを実行

セルの実行中だけ学習します。制限回避・keep-alive・バックグラウンドワーカーはありません。
`BATCH_SIZE` と `GRAD_ACCUM` は自宅と異なっていて構いません。
`precision="auto"` は対応GPUでBF16、それ以外でFP32。T4では `"fp16"` も選べます。
切断後は同じ `WORK_DIR` を保持して再実行すると、保存済み差分の再送を試みます。
Colab VM自体が失われた場合、未送信作業は復元できません。受理済みモデルはcoordinatorに残ります。

In [ ]:
from wsda.worker import run_worker

BATCH_SIZE, GRAD_ACCUM, JOBS = 1, 2, 10
WORK_DIR = BASE / "wsda_colab_outbox"
if not I_HAVE_AUTHORIZED_DISTRIBUTED_COMPUTE:
    raise RuntimeError("実行条件を確認してください")
results = run_worker(
    server=SERVER, data=str(DATA), worker_id="colab", token=TOKEN,
    jobs=JOBS, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM,
    device="auto", precision="auto", wire="int8", work_dir=WORK_DIR,
    shard_index=1, num_shards=2,
)
print({"accepted": sum(r["accepted"] for r in results), "jobs": len(results)})


## 5. 受理済みモデルの取得

自宅で以下を実行してexportします。validation lossで品質を確認してください。

```bash
docker compose exec coordinator wsda export --state /state/coordinator --out /state/export
docker compose cp coordinator:/state/export ./export
wsda evaluate --checkpoint export --data data
```

このモデルはゼロから学習するカスタム実装であり、既存LLMの重みを自動で変換するものではありません。
SDAは未融合の研究用実装です。小規模設定で確認してからモデルを拡張してください。
